# 📘 Notebook 2 - Catálogo de Dados e Análise da Qualidade


---

## 🎲 2.1. Esquema de dados, conforme obtidos na fonte

Inicialmente observamos os arquivos CSV que compõem o dataset:

In [0]:
base_path = "/tmp/zip_data"
#display(dbutils.fs.ls("dbfs:/tmp/zip_data"))
display(dbutils.fs.ls(f"dbfs:{base_path}"))

path,name,size,modificationTime
dbfs:/tmp/zip_data/data_dictionary.csv,data_dictionary.csv,4067,1743883158000
dbfs:/tmp/zip_data/encounters.csv,encounters.csv,7647591,1743883157000
dbfs:/tmp/zip_data/organizations.csv,organizations.csv,163,1743883158000
dbfs:/tmp/zip_data/patients.csv,patients.csv,221122,1743883157000
dbfs:/tmp/zip_data/payers.csv,payers.csv,1030,1743883158000
dbfs:/tmp/zip_data/procedures.csv,procedures.csv,8712632,1743883156000


São seis arquivos, sendo que um deles é o dicionário de dados. Vamos analisar seu conteúdo:

In [0]:
df_dictionary = spark.read.option("header", True).csv(f"dbfs:{base_path}/data_dictionary.csv")
df_dictionary.display()

Table,Field,Description
encounters,null,Patient encounter data
encounters,Id,Primary Key. Unique Identifier of the encounter.
encounters,Start,The date and time (iso8601 UTC Date (yyyy-MM-dd'T'HH:mm'Z')) the encounter started
encounters,Stop,The date and time (iso8601 UTC Date (yyyy-MM-dd'T'HH:mm'Z')) the encounter concluded
encounters,Patient,Foreign key to the Patient.
encounters,Organization,Foreign key to the Organization.
encounters,Payer,Foreign key to the Payer.
encounters,EncounterClass,"The class of the encounter, such as ambulatory, emergency, inpatient, wellness, or urgentcare"
encounters,Code,Encounter code from SNOMED-CT
encounters,Description,Description of the type of encounter.


A partir da análise do dicionário de dados fornecido, com indicação das tabelas, campos e chaves, pudemos montar o diagrama abaixo, que ilustra o modelo lógico do banco de dados relacional capaz de aramzenar os dados do dataset:

![](https://raw.githubusercontent.com/cristianofanchin/puc-rio/main/engenhariadados/diagrama_bronze.png)

## 📚 2.2. Catálogo dos dados

Este catálogo documenta as principais entidades e atributos utilizadas no projeto, com informações de chaves primárias, estrangeiras e descrições, com o objetivo de promover rastreabilidade e compreensão semântica dos dados.

---

## 🗃️ Tabela: `patients`

| Campo       | Tipo       | Descrição                                                                 |
|-------------|------------|---------------------------------------------------------------------------|
| Id          | String     | **PK.** Identificador único do paciente                                   |
| BirthDate   | Date       | Data de nascimento                                                        |
| DeathDate   | Date       | Data de falecimento (se aplicável)                                        |
| Prefix      | String     | Prefixo do nome (ex: Sr., Sra., Dr.)                                      |
| First       | String     | Primeiro nome                                                             |
| Last        | String     | Sobrenome                                                                 |
| Suffix      | String     | Sufixo do nome (ex: PhD, Jr.)                                             |
| Maiden      | String     | Nome de solteiro                                                          |
| Marital     | String     | Estado civil (M = casado, S = solteiro)                                   |
| Race        | String     | Raça principal                                                            |
| Ethnicity   | String     | Etnia principal                                                           |
| Gender      | String     | Gênero (M ou F)                                                           |
| BirthPlace  | String     | Cidade de nascimento                                                      |
| Address     | String     | Endereço do paciente                                                      |
| City        | String     | Cidade                                                                     |
| State       | String     | Estado                                                                    |
| County      | String     | Condado                                                                   |
| Zip         | String     | Código postal                                                             |
| Lat         | String     | Latitude                                                                  |
| Lon         | String     | Longitude                                                                 |

---

##🗃️ Tabela: `encounters`

| Campo              | Tipo     | Descrição                                                                                  |
|--------------------|----------|----------------------------------------------------------------------------------------------|
| Id                 | String   | **PK.** Identificador único do encontro                                                     |
| Start              | DateTime | Data e hora de início                                                                       |
| Stop               | DateTime | Data e hora de término                                                                      |
| Patient            | String   | **FK → patients(Id).** Paciente participante                                                |
| Organization       | String   | **FK → organizations(Id).** Organização prestadora                                          |
| Payer              | String   | **FK → payers(Id).** Pagador responsável                                                    |
| EncounterClass     | String   | Tipo de encontro (ambulatorial, emergência, internação etc.)                                |
| Code               | String   | Código SNOMED-CT do tipo de encontro                                                        |
| Description        | String   | Descrição do encontro                                                                       |
| Base_Encounter_Cost| Float    | Custo base do encontro (sem itens adicionais)                                               |
| Total_Claim_Cost   | Float    | Custo total incluindo todos os itens                                                        |
| Payer_Coverage     | Float    | Valor coberto pelo pagador                                                                  |
| ReasonCode         | String   | Código da condição alvo do encontro                                                         |
| ReasonDescription  | String   | Descrição da razão do encontro                                                              |

---

##🗃️ Tabela: `organizations`

| Campo   | Tipo    | Descrição                                        |
|---------|---------|--------------------------------------------------|
| Id      | String  | **PK.** Identificador da organização             |
| Name    | String  | Nome da organização                              |
| Address | String  | Endereço                                         |
| City    | String  | Cidade                                           |
| State   | String  | Estado                                           |
| Zip     | String  | CEP                                              |
| Lat     | String  | Latitude                                         |
| Lon     | String  | Longitude                                        |

---

##🗃️ Tabela: `payers`

| Campo             | Tipo   | Descrição                                 |
|-------------------|--------|--------------------------------------------|
| Id                | String | **PK.** Identificador do pagador           |
| Name              | String | Nome do pagador                            |
| Address           | String | Endereço                                   |
| City              | String | Cidade                                     |
| State_Headquartered | String | Estado da sede                            |
| Zip               | String | Código postal                              |
| Phone             | String | Telefone                                   |

---

##🗃️ Tabela: `procedures`

| Campo            | Tipo     | Descrição                                                                 |
|------------------|----------|---------------------------------------------------------------------------|
| Start            | DateTime | Início do procedimento                                                    |
| Stop             | DateTime | Fim do procedimento (se aplicável)                                        |
| Patient          | String   | **FK → patients(Id).** Paciente submetido                                 |
| Encounter        | String   | **FK → encounters(Id).** Encontro relacionado                             |
| Code             | String   | Código SNOMED-CT do procedimento                                           |
| Description      | String   | Descrição textual do procedimento                                          |
| Base_Cost        | Float    | Custo do procedimento                                                     |
| ReasonCode       | String   | Código da razão para o procedimento                                        |
| ReasonDescription| String   | Descrição da razão                                                        |

---


## 📊 2.3. Qualidade dos dados

### 📋 Informações básicas e verificação de nulos

Iremos inicialmente extrair e imprimir informações de cada arquivo CSV que compõem o dataset e verificar a existência de nulos.

Faremos isso carregando um CSV por vez para dentro de um datafreme PySpark e usando recursos de apreentação do Schema e um "select" para contar os nulos presentes em cada coluna.

In [0]:
#Ver resumo por arquivo individual a partir da estrutura dbfs:/tmp/zip_data

from pyspark.sql.functions import col, sum
arquivos = dbutils.fs.ls("dbfs:/tmp/zip_data/")
for file in [f.name for f in arquivos]:
    if file.endswith(".csv"):
        full_path = f"dbfs:/tmp/zip_data/{file}"
        df = spark.read.option("header", True).csv(full_path)
        print(f"📄 Arquivo: {file} | {df.count()} linhas")
        df.printSchema()
        print("\nVerificação de Nulos:")
        df.select([sum(col(c).isNull().cast("int")).alias(c) for c in df.columns]).show()

📄 Arquivo: data_dictionary.csv | 65 linhas
root
 |-- Table: string (nullable = true)
 |-- Field: string (nullable = true)
 |-- Description: string (nullable = true)


Verificação de Nulos:
+-----+-----+-----------+
|Table|Field|Description|
+-----+-----+-----------+
|    0|    5|          0|
+-----+-----+-----------+

📄 Arquivo: encounters.csv | 27891 linhas
root
 |-- Id: string (nullable = true)
 |-- START: string (nullable = true)
 |-- STOP: string (nullable = true)
 |-- PATIENT: string (nullable = true)
 |-- ORGANIZATION: string (nullable = true)
 |-- PAYER: string (nullable = true)
 |-- ENCOUNTERCLASS: string (nullable = true)
 |-- CODE: string (nullable = true)
 |-- DESCRIPTION: string (nullable = true)
 |-- BASE_ENCOUNTER_COST: string (nullable = true)
 |-- TOTAL_CLAIM_COST: string (nullable = true)
 |-- PAYER_COVERAGE: string (nullable = true)
 |-- REASONCODE: string (nullable = true)
 |-- REASONDESCRIPTION: string (nullable = true)


Verificação de Nulos:
+---+-----+----+------

**Constatações**

- Na tabela **patients**, vamos a existência de nulos em campos aceitáveis, como por exemplo "deathdata" (null = paciente está vivo), sufixo de nome, código postal e nome de solteira. Para apenas um registro, está nulo o campo "marital" (estado civil). Concluímos que a qualidade desses dados é alta.

- Na tabela **payers**, só há campos nulos para o registro que indica a ausência de um segurador, o que é o esperado.

- Na tabela **procedures**, há 36.945	registros como campo "ReasonCode" nulo, de um total 47.701 registros, o que representa 77% de ocorrências.
Esse campo abriga o diagnóstico que motivou o procedimento hospitalar e, dada a incidência de nulos, vemos que o hospital em questão não fez uso consistente desse campo no registro dos procedimentos.
Esse é um motivo para descartar o uso desse campo em eventuais estatísticas desse dataset.

- Na tabela **encounters**, vemos a mesma situação para o campo "ReasonCode", em que 70% dos registros (19.451 de 27.891) não contém valores nesse campo. Os demais campos de encounter não possuem nulos.

- Por fim, a tabela **organizations** está com todos os campos completos no único registro que abriga.


### 🗺 Verificação da qualidade dos dados de Latitude e Longitude

Nessa seção faremos uma análise visual dos dados de latitude e longitude que representam o endereço dos pacientes. Usaremos o recurso de construção de mapa interativo da biblioteca _folium_. 

In [0]:
pip install folium

Python interpreter will be restarted.
  Using cached folium-0.19.5-py2.py3-none-any.whl (110 kB)
  Using cached branca-0.8.1-py3-none-any.whl (26 kB)
  Using cached xyzservices-2025.1.0-py3-none-any.whl (88 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl (134 kB)
  Attempting uninstall: jinja2
    Found existing installation: Jinja2 2.11.3
    Not uninstalling jinja2 at /databricks/python3/lib/python3.9/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-1b4edfa1-d29d-4031-9353-aa080041746f
    Can't uninstall 'Jinja2'. No files were found to uninstall.
Python interpreter will be restarted.


In [0]:
import folium
import pandas as pd

base_path = "dbfs:/tmp/zip_data"

# Ler o CSV diretamente do DBFS
df = spark.read.option("header", True).csv(f"{base_path}/patients.csv")

# Agrupar por LAT e LON e contar ocorrências
df_grouped = df.groupBy("LAT", "LON").count()

# Converter para Pandas para visualização com folium
df_pandas = df_grouped.toPandas()

# Converter LAT e LON para float
df_pandas['LAT'] = df_pandas['LAT'].astype(float)
df_pandas['LON'] = df_pandas['LON'].astype(float)

# Etapa 4: Criar o mapa centralizado
mapa = folium.Map(location=[df_pandas['LAT'].mean(), df_pandas['LON'].mean()], zoom_start=5)

# Adicionar bolhas proporcionais à contagem
for _, row in df_pandas.iterrows():
    folium.CircleMarker(
        location=[row['LAT'], row['LON']],
        radius=3 + row['count']**0.5,  # aumenta bolha com sqrt do count
        #color='crimson',
        color='blue',
        fill=True,
        fill_color='blue',
        fill_opacity=0.6,
        popup=f"Ocorrências: {row['count']}"
    ).add_to(mapa)

# Exibir mapa (em notebook) ou salvar como HTML
#mapa.save("mapa_bolhas.html")
#mapa  # Em notebooks interativos



In [0]:
# Mostrar o mapa
mapa

Make this Notebook Trusted to load map: File -> Trust Notebook <iframe srcdoc="<!DOCTYPE html>
<html>
<head>
 
 <meta http-equiv="content-type" content="text/html; charset=UTF-8" />
 
 <script>
 L_NO_TOUCH = false;
 L_DISABLE_3D = false;
 </script>
 
 <style>html, body {width: 100%;height: 100%;margin: 0;padding: 0;}</style>
 <style>#map {position:absolute;top:0;bottom:0;right:0;left:0;}</style>
 <script src="https://cdn.jsdelivr.net/npm/leaflet@1.9.3/dist/leaflet.js"></script>
 <script src="https://code.jquery.com/jquery-3.7.1.min.js"></script>
 <script src="https://cdn.jsdelivr.net/npm/bootstrap@5.2.2/dist/js/bootstrap.bundle.min.js"></script>
 <script src="https://cdnjs.cloudflare.com/ajax/libs/Leaflet.awesome-markers/2.0.2/leaflet.awesome-markers.js"></script>
 <link rel="stylesheet" href="https://cdn.jsdelivr.net/npm/leaflet@1.9.3/dist/leaflet.css"/>
 <link rel="stylesheet" href="https://cdn.jsdelivr.net/npm/bootstrap@5.2.2/dist/css/bootstrap.min.css"/>
 <link rel="stylesheet" href="https://netdna.bootstrapcdn.com/bootstrap/3.0.0/css/bootstrap-glyphicons.css"/>
 <link rel="stylesheet" href="https://cdn.jsdelivr.net/npm/@fortawesome/fontawesome-free@6.2.0/css/all.min.css"/>
 <link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/Leaflet.awesome-markers/2.0.2/leaflet.awesome-markers.css"/>
 <link rel="stylesheet" href="https://cdn.jsdelivr.net/gh/python-visualization/folium/folium/templates/leaflet.awesome.rotate.min.css"/>
 
 <meta name="viewport" content="width=device-width,
 initial-scale=1.0, maximum-scale=1.0, user-scalable=no" />
 <style>
 #map_84de4c80ef88c8fe99cc27b78f8ef52f {
 position: relative;
 width: 100.0%;
 height: 100.0%;
 left: 0.0%;
 top: 0.0%;
 }
 .leaflet-container { font-size: 1rem; }
 </style>
 
</head>
<body>
 
 
 <div class="folium-map" id="map_84de4c80ef88c8fe99cc27b78f8ef52f" ></div>
 
</body>
<script>
 
 
 var map_84de4c80ef88c8fe99cc27b78f8ef52f = L.map(
 "map_84de4c80ef88c8fe99cc27b78f8ef52f",
 {
 center: [42.33735872413591, -71.02752362432238],
 crs: L.CRS.EPSG3857,
 ...{
 "zoom": 5,
 "zoomControl": true,
 "preferCanvas": false,
}

 }
 );

 

 
 
 var tile_layer_0707e5f427dad20837b7a05ef5514b11 = L.tileLayer(
 "https://tile.openstreetmap.org/{z}/{x}/{y}.png",
 {
 "minZoom": 0,
 "maxZoom": 19,
 "maxNativeZoom": 19,
 "noWrap": false,
 "attribution": "\u0026copy; \u003ca href=\"https://www.openstreetmap.org/copyright\"\u003eOpenStreetMap\u003c/a\u003e contributors",
 "subdomains": "abc",
 "detectRetina": false,
 "tms": false,
 "opacity": 1,
}

 );
 
 
 tile_layer_0707e5f427dad20837b7a05ef5514b11.addTo(map_84de4c80ef88c8fe99cc27b78f8ef52f);
 
 
 var circle_marker_8f57c59694355692f454cdec22bdf51a = L.circleMarker(
 [42.33832106011372, -71.15545553],
 {"bubblingMouseEvents": true, "color": "blue", "dashArray": null, "dashOffset": null, "fill": true, "fillColor": "blue", "fillOpacity": 0.6, "fillRule": "evenodd", "lineCap": "round", "lineJoin": "round", "opacity": 1.0, "radius": 4.0, "stroke": true, "weight": 3}
 ).addTo(map_84de4c80ef88c8fe99cc27b78f8ef52f);
 
 
 var popup_779418fc9cfc7529899f49e95a5027d2 = L.popup({
 "maxWidth": "100%",
});

 
 
 var html_cb8ee83632fb7a237757d9afe5c31f96 = $(`<div id="html_cb8ee83632fb7a237757d9afe5c31f96" style="width: 100.0%; height: 100.0%;">Ocorrências: 1.0</div>`)[0];
 popup_779418fc9cfc7529899f49e95a5027d2.setContent(html_cb8ee83632fb7a237757d9afe5c31f96);
 
 

 circle_marker_8f57c59694355692f454cdec22bdf51a.bindPopup(popup_779418fc9cfc7529899f49e95a5027d2)
 ;

 
 
 
 var circle_marker_8d24aa5179054385b03727804ad48c0c = L.circleMarker(
 [42.35365691303956, -71.03266516],
 {"bubblingMouseEvents": true, "color": "blue", "dashArray": null, "dashOffset": null, "fill": true, "fillColor": "blue", "fillOpacity": 0.6, "fillRule": "evenodd", "lineCap": "round", "lineJoin": "round", "opacity": 1.0, "radius": 4.0, "stroke": true, "weight": 3}
 ).addTo(map_84de4c80ef88c8fe99cc27b78f8ef52f);
 
 
 var popup_3567495e69ba44ac9f9f46543f40bd07 = L.popup({
 "maxWidt

![](https://raw.githubusercontent.com/cristianofanchin/puc-rio/main/engenhariadados/mapa1.png)

Conforme podemos observar, alguns desses dados estão certamente inconsistentes, pois estão no meio da Baía de Massachusetts (e alguns até na pista do aeroporto de Boston)!

Mesmo assim, optamos por manter esses dados na base, pois sua limpeza seria de complexidade mais elevada, fugindo ao propósito desse MVP.

### 🧹 Demais análises de qualidade dos dados

Promoveremos nesse trabalho algumas outras análises da qualidade dos dados durante a construção da camada Silver, para que haja a oportunidade de usar a linguagem SQL tanto para as consultas quanto para as limpezas dos dados que forem necessárias.

Como até esse passo os dados não estão ainda carregados em banco de dados, apresentaremos e discorreremos sobre esseas outras análises na próxima seção:

- Busca de inconsistências nos campos de data de nascimento e morte de pacientes;

- Busca de pacientes cujo nome contenha caracteres inválidos;

- Busca de inconsistências nos campos de data e hora de início e fim de atendimentos;

- Verificação de variáveis categóricas, em busca de inconsistências:
  - Encounters ➡ EncounterClass
  - Encounters ➡ ReasonDescription
  - Patient ➡ Marital
  - Patient ➡ Gender
  - Patient ➡ Race
   - Patient ➡ Ethnicity

<br>

> Siga para o próximo notebook [🚀](https://github.com/cristianofanchin/puc-rio/blob/main/engenhariadados/3-ETL_Bronze_Silver_Gold.ipynb)

> [◀](https://github.com/cristianofanchin/puc-rio/blob/main/engenhariadados/README.md) Volte para o início 
